# HyDRA v7 — Visualizer Dimension 10
**Multi-dimensional hyperbolic space visualization**

> Rotação simultânea em 4 eixos W. Pacemakers {4,8,11} como anéis dimensionais distintos.
> Spiral L_align conectando manifolds student↔teacher. Composite Kuramoto gating visual.
> Background MP4 recording contínuo. Vista 360°.

In [ ]:
"""Cell 1 — Mount + Config"""
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path

RUN_MODE = "v7"  # "v7" | "v6" | "v5" | "v4" | "v3" | "magnetic"

ROOTS = {
    "magnetic": Path("/content/drive/MyDrive/HydraPaper_Magnetic"),
    "v3":       Path("/content/drive/MyDrive/HydraPaper_v3"),
    "v4":       Path("/content/drive/MyDrive/HydraPaper_v4"),
    "v5":       Path("/content/drive/MyDrive/HydraPaper_v5"),
    "v6":       Path("/content/drive/MyDrive/HydraPaper_v6"),
    "v7":       Path("/content/drive/MyDrive/HydraPaper_v7"),
}
RUN_TAGS = {
    "magnetic": "hydra_magnetic_v1",
    "v3":       "hydra_v3_temporal_binding",
    "v4":       "hydra_v4_cgt_binding",
    "v5":       "hydra_v5_anosov_topo",
    "v6":       "hydra_v6_gcm",
    "v7":       "hydra_v7_gcm",
}
LABELS = {
    "magnetic": "HyDRA Magnetic",
    "v3":       "HyDRA v3",
    "v4":       "HyDRA v4 · CGT",
    "v5":       "HyDRA v5 · Anosov",
    "v6":       "HyDRA v6 · GCM",
    "v7":       "HyDRA v7 · GCM Optimal",
}

# v7 pacemaker layers
PACEMAKER_LAYERS = {4, 8, 11}  # syntactic · semantic · aggregation
PACEMAKER_COLORS = {4: "#00e5ff", 8: "#ffd600", 11: "#e040fb"}
PACEMAKER_NAMES  = {4: "SYNTACTIC", 8: "SEMANTIC", 11: "AGGREGATE"}

PAPER_ROOT    = ROOTS[RUN_MODE]
RUN_TAG       = RUN_TAGS[RUN_MODE]
LOG_DIR       = PAPER_ROOT / "logs" / RUN_TAG
TRAIN_CSV     = LOG_DIR / "training_metrics.csv"
VAL_CSV       = LOG_DIR / "val_metrics.csv"
BINDING_CSV   = LOG_DIR / "binding_metrics.csv"
SNAPSHOT_PATH = LOG_DIR / "embeddings_snapshot.json"
GW_CSV        = LOG_DIR / "gw_metrics.csv"
TIMELAPSE_DIR = PAPER_ROOT / "timelapse"
TIMELAPSE_DIR.mkdir(parents=True, exist_ok=True)
RUN_LABEL     = LABELS[RUN_MODE]

print(f"Mode : {RUN_MODE.upper()} — {RUN_LABEL}")
for n,p in [("Train",TRAIN_CSV),("Val",VAL_CSV),("Binding",BINDING_CSV),
            ("Snapshot",SNAPSHOT_PATH),("GW",GW_CSV)]:
    print(f"  {n:<8}: {'✅' if p.exists() else '⏳'}")


Mounted at /content/drive
Mode : V7 — HyDRA v7 · GCM Optimal
  Train   : ⏳
  Val     : ⏳
  Binding : ⏳
  Snapshot: ⏳
  GW      : ⏳


In [ ]:
"""Cell 2 — Utilities"""
import csv, json as _json, math, time

def read_csv(p):
    try: return list(csv.DictReader(open(p)))
    except: return []

def latest_metrics(train, val):
    t = train[-1] if train else {}
    v = val[-1]   if val   else {}
    return {
        "step":     int(t.get("step",0) or 0),
        "ppl":      float(t.get("ppl",9999) or 9999),
        "rdc":      float(t.get("rdc_ema",0) or 0),
        "rad":      float(t.get("mean_radius",1.5) or 1.5),
        "div":      float(t.get("diversity",0) or 0),
        "logit_std":float(t.get("logit_std",0) or 0),
        "w_ent":    float(t.get("w_entropy",0) or 0),
        "l_hidden": float(t.get("l_hidden",0) or 0),
        "k_eff":    float(t.get("k_eff",0.3) or 0.3),
        "l_align":  float(t.get("l_align",0) or 0),
        "aux_loss": float(t.get("aux_loss",0) or 0),
        "val_ppl":  float(v.get("val_ppl",9999) or 9999),
        "val_step": int(v.get("step",0) or 0),
    }

def latest_binding(rows):
    b = rows[-1] if rows else {}
    try:    phases = _json.loads(b.get("phases_json","[]"))
    except: phases = []
    return {
        "order_param": float(b.get("order_param",0) or 0),
        "phases":      phases,
    }

def load_snapshot():
    if not SNAPSHOT_PATH.exists(): return {}
    try: return _json.loads(SNAPSHOT_PATH.read_text())
    except: return {}

print("✅ Utilities ready")


✅ Utilities ready


In [ ]:
VIZ_HTML = r"""
<!DOCTYPE html><html><head><meta charset="utf-8">
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne+Mono&family=Syncopate:wght@400;700&display=swap');
*{box-sizing:border-box;margin:0;padding:0}
body{background:#000;overflow:hidden}
#root{width:100%;height:{{HEIGHT}}px;position:relative;
      background:radial-gradient(ellipse at 40% 35%,#020510 0%,#000105 60%,#000000 100%);
      border-radius:12px;overflow:hidden;font-family:'Syne Mono',monospace}
canvas{display:block;width:100%;height:100%}
/* HUD */
.hud{position:absolute;top:16px;left:18px;pointer-events:none;z-index:20;
     line-height:1.9;font-size:10px;color:rgba(180,210,255,0.82)}
.title{font-family:'Syncopate',sans-serif;font-size:15px;font-weight:700;
       letter-spacing:.18em;color:#fff;text-shadow:0 0 25px rgba(80,140,255,.9);
       margin-bottom:10px;display:block}
.dim-badge{font-family:'Syncopate',sans-serif;font-size:9px;letter-spacing:.2em;
           color:rgba(0,229,255,.6);margin-bottom:8px;display:block}
.v{color:#39ffa0}.g{color:#ffd600}.b{color:#00e5ff}.p{color:#e040fb}
.r{color:#ff4060}.dim{color:#1e2d50;font-size:8.5px}
/* Pacemaker legend */
.pac{position:absolute;top:16px;right:18px;pointer-events:none;z-index:20;
     font-size:8.5px;text-align:right;line-height:2.2}
.pac-item{display:flex;align-items:center;justify-content:flex-end;gap:6px}
.pac-dot{width:8px;height:8px;border-radius:50%;display:inline-block}
/* Gamma badge */
.gamma{position:absolute;bottom:46px;right:18px;
       font-family:'Syncopate',sans-serif;font-size:24px;font-weight:700;
       color:rgba(224,64,251,.72);pointer-events:none;z-index:20;
       text-shadow:0 0 30px rgba(200,50,255,.6)}
/* K_eff badge */
.keff{position:absolute;bottom:78px;right:18px;
      font-family:'Syncopate',sans-serif;font-size:11px;
      color:rgba(0,229,255,.65);pointer-events:none;z-index:20}
/* L_align badge */
.lalign{position:absolute;bottom:98px;right:18px;
        font-family:'Syncopate',sans-serif;font-size:10px;
        color:rgba(57,255,160,.55);pointer-events:none;z-index:20}
/* W-axes indicator */
.waxes{position:absolute;bottom:16px;left:18px;pointer-events:none;z-index:20;
       font-size:8px;color:#0d1a35;line-height:1.9}
/* Controls */
.ctrl{position:absolute;bottom:16px;right:18px;font-size:8px;
      color:#0d1a35;pointer-events:none;z-index:20;text-align:right;line-height:1.9}
/* Scan line overlay */
.scan{position:absolute;top:0;left:0;width:100%;height:100%;pointer-events:none;z-index:5;
      background:repeating-linear-gradient(0deg,transparent,transparent 2px,
        rgba(0,0,0,.04) 2px,rgba(0,0,0,.04) 4px);border-radius:12px}
</style></head><body>
<div id="root">
<canvas id="c10d"></canvas>
<div class="scan"></div>

<div class="hud">
  <span class="title">{{LABEL}}</span>
  <span class="dim-badge">DIMENSION 10 · LORENTZ ℍⁿ</span>
  <div>STEP &nbsp; {{STEP}} / 200,000</div>
  <div class="v">PPL &nbsp;&nbsp; {{PPL}}</div>
  <div class="v">RDC &nbsp;&nbsp; {{RDC}} ✓</div>
  <div class="g">RAD &nbsp;&nbsp; {{RAD}}</div>
  <div class="g">LSTD &nbsp; {{LSTD}}</div>
  <div class="b">DIV &nbsp;&nbsp; {{DIV}}</div>
  <div class="b">W·ENT &nbsp;{{WENT}}</div>
  <div class="p">Γ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; {{GAMMA}}</div>
  <div class="b">K_eff &nbsp; {{KEFF}}</div>
  <div class="v">L_align {{LALIGN}}</div>
  <div class="dim" style="margin-top:4px">val {{VALPPL}} @ {{VALSTEP}}</div>
</div>

<div class="pac">
  <div class="pac-item"><span style="font-size:8px;color:#00e5ff">L4 SYNTACTIC</span>
    <span class="pac-dot" style="background:#00e5ff;box-shadow:0 0 6px #00e5ff"></span></div>
  <div class="pac-item"><span style="font-size:8px;color:#ffd600">L8 SEMANTIC</span>
    <span class="pac-dot" style="background:#ffd600;box-shadow:0 0 6px #ffd600"></span></div>
  <div class="pac-item"><span style="font-size:8px;color:#e040fb">L11 AGGREGATE</span>
    <span class="pac-dot" style="background:#e040fb;box-shadow:0 0 6px #e040fb"></span></div>
  <div style="color:#0d1a35;font-size:7.5px;margin-top:4px;text-align:right" id="fps-ctr">-- fps</div>
</div>

{{GAMMA_BADGE}}
{{KEFF_BADGE}}
{{LALIGN_BADGE}}

<div class="waxes">
  W₁ = sin(θ_kuramoto)<br>
  W₂ = cos(φ_semantic)<br>
  W₃ = tanh(r_lorentz)<br>
  W₄ = Γ(t) · sin(ωt)<br>
  <span id="w-angles">θ₁=0.000</span>
</div>

<div class="ctrl">
  DRAG orbit · SCROLL zoom<br>
  SPACE pause W-rot · R reset<br>
  1-4 W-axis speed
</div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
(function(){
const root   = document.getElementById("root");
const canvas = document.getElementById("c10d");
const W = root.clientWidth, H = {{HEIGHT}};

const renderer = new THREE.WebGLRenderer({canvas,antialias:true,alpha:false});
renderer.setSize(W,H);
renderer.setPixelRatio(Math.min(window.devicePixelRatio,2));
renderer.setClearColor(0x000000,1);

const scene  = new THREE.Scene();
scene.fog    = new THREE.FogExp2(0x000105, 0.038);
const camera = new THREE.PerspectiveCamera(44,W/H,0.01,300);
camera.position.set(0,3,10);

// ── DATA ──────────────────────────────────────────────────────────────────
const TOKEN_DATA  = {{TOKEN_DATA}};
const PHASE_DATA  = {{PHASE_DATA}};
const TRAIL_DATA  = {{TRAIL_DATA}};
const GW_DATA     = {{GW_DATA}};
const GAMMA       = {{GAMMA_VAL}};
const CUR_RAD     = {{CUR_RAD}};
const CUR_RDC     = {{CUR_RDC}};
const CUR_LSTD    = {{CUR_LSTD}};
const KEFF        = {{KEFF_VAL}};
const IS_V7       = {{IS_V7}};
const IS_V3456    = {{IS_V3456}};

// ── 4-AXIS W ROTATION ─────────────────────────────────────────────────────
let wAngles = [0,0,0,0];
let wSpeeds = [0.0055, 0.0038, 0.0021, 0.0013];
let wPaused = false;

// 4D→3D perspective projection with 4 W-axes
function project10D(x,y,z, w1,w2,w3,w4) {
  const wDist = 3.2;
  const w = w1*0.5 + w2*0.3 + w3*0.15 + w4*0.05;
  const s = wDist / Math.max(wDist - w, 0.1);
  return [x*s, y*s, z*s];
}
function rotateW1(x,y,z,w,a){const c=Math.cos(a),s=Math.sin(a);return[x*c-w*s,y,z,x*s+w*c];}
function rotateW2(x,y,z,w,a){const c=Math.cos(a),s=Math.sin(a);return[x,y*c-w*s,z,y*s+w*c];}
function rotateW3(x,y,z,w,a){const c=Math.cos(a),s=Math.sin(a);return[x,y,z*c-w*s,z*s+w*c];}

// ── STARFIELD ─────────────────────────────────────────────────────────────
{
  const N=1200, sp=new Float32Array(N*3), sc=new Float32Array(N*3);
  for(let i=0;i<N;i++){
    const r=40+Math.random()*120;
    const a=Math.random()*Math.PI*2, b=Math.random()*Math.PI*2;
    sp[i*3]=Math.cos(a)*Math.cos(b)*r;
    sp[i*3+1]=Math.sin(b)*r*0.6;
    sp[i*3+2]=Math.sin(a)*Math.cos(b)*r;
    const bright=0.2+Math.random()*0.5;
    sc[i*3]=bright*0.7; sc[i*3+1]=bright*0.8; sc[i*3+2]=bright;
  }
  const sg=new THREE.BufferGeometry();
  sg.setAttribute("position",new THREE.BufferAttribute(sp,3));
  sg.setAttribute("color",new THREE.BufferAttribute(sc,3));
  scene.add(new THREE.Points(sg,new THREE.PointsMaterial({size:0.15,vertexColors:true,transparent:true,opacity:0.55})));
}

// ── SOFT HYPERBOLOID — 3 NESTED SHELLS ───────────────────────────────────
// Shell 1: student manifold (warm amber)
// Shell 2: transition zone (cyan)
// Shell 3: teacher manifold (dim blue)
function makeSoftShell(rad, col, opacity, rings=14, merid=56) {
  const g=new THREE.Group();
  for(let i=0;i<rings;i++){
    const y=(i/(rings-1)-0.5)*5.5;
    const r=Math.cosh(y*0.38)*rad;
    const tub=0.005+0.004*Math.abs(Math.sin(i/rings*Math.PI));
    const op=opacity*(0.35+0.65*(1-Math.abs(i-(rings/2))/(rings/2)));
    const mesh=new THREE.Mesh(
      new THREE.TorusGeometry(r,tub,6,merid),
      new THREE.MeshBasicMaterial({color:col,transparent:true,opacity:op,side:THREE.DoubleSide}));
    mesh.rotation.x=Math.PI/2; mesh.position.y=y;
    g.add(mesh);
  }
  return g;
}
// Student
const shellS = makeSoftShell(CUR_RAD, 0xffb84f, 0.50);
scene.add(shellS);
const shellSg= makeSoftShell(CUR_RAD, 0xffb84f, 0.08, 6, 24);
shellSg.scale.set(1.05,1.02,1.05); scene.add(shellSg);
// Transition (L_align zone)
const shellT = makeSoftShell(CUR_RAD*1.18, 0x00e5ff, 0.10, 8, 32);
scene.add(shellT);
// Teacher (dim outer)
const shellP = makeSoftShell(CUR_RAD*1.40, 0x1a3a6a, 0.06, 5, 20);
scene.add(shellP);

// Meridians
for(let j=0;j<40;j++){
  const ang=(j/40)*Math.PI*2;
  const pts=[];
  for(let t=-3;t<=3;t+=0.06){
    const r=Math.cosh(t*0.42)*CUR_RAD;
    pts.push(new THREE.Vector3(Math.cos(ang)*r,t*0.5,Math.sin(ang)*r));
  }
  scene.add(new THREE.Line(new THREE.BufferGeometry().setFromPoints(pts),
    new THREE.LineBasicMaterial({color:0x030815,transparent:true,opacity:0.38})));
}

// ── PACEMAKER RINGS — 3 DISTINCT LAYERS {4,8,11} ──────────────────────────
// Each at different height + radius representing binding depth
const pacRings=[];
const pacData=[
  {y:-1.2, r:CUR_RAD*0.82, col:0x00e5ff, glow:0x004055, label:"L4"},
  {y: 0.0, r:CUR_RAD*0.96, col:0xffd600, glow:0x404000, label:"L8"},
  {y: 1.2, r:CUR_RAD*1.08, col:0xe040fb, glow:0x400055, label:"L11"},
];
pacData.forEach((pd,k)=>{
  // Primary ring
  const rm=new THREE.Mesh(
    new THREE.TorusGeometry(pd.r,0.016,8,96),
    new THREE.MeshBasicMaterial({color:pd.col,transparent:true,opacity:0.55}));
  rm.rotation.x=Math.PI/2; rm.position.y=pd.y; scene.add(rm);
  // Glow ring
  const rg=new THREE.Mesh(
    new THREE.TorusGeometry(pd.r,0.10,6,64),
    new THREE.MeshBasicMaterial({color:pd.glow,transparent:true,opacity:0.07}));
  rg.rotation.x=Math.PI/2; rg.position.y=pd.y; scene.add(rg);
  // Inner pulse ring
  const ri=new THREE.Mesh(
    new THREE.TorusGeometry(pd.r*0.94,0.008,6,64),
    new THREE.MeshBasicMaterial({color:pd.col,transparent:true,opacity:0.22}));
  ri.rotation.x=Math.PI/2; ri.position.y=pd.y; scene.add(ri);
  pacRings.push({rm,rg,ri,col:pd.col,y:pd.y,r:pd.r,phase:k*Math.PI*2/3});
});

// ── TOKEN PARTICLES — 10D positions ──────────────────────────────────────
let tokenPos4D=[];
const tokenMeshes=[];

function buildTokens(){
  const N = TOKEN_DATA.length>0 ? TOKEN_DATA.length : 650;
  const pos=new Float32Array(N*3), col=new Float32Array(N*3);

  for(let i=0;i<N;i++){
    let x,y,z,w1,w2,w3,w4;
    if(TOKEN_DATA.length>0){
      const t=TOKEN_DATA[i]||{x:0,y:0,z:0,sub:"A"};
      x=t.x; y=t.y; z=t.z;
      const ph = PHASE_DATA.length>0 ? PHASE_DATA[i%PHASE_DATA.length] : Math.random()*Math.PI*2;
      w1=Math.sin(ph)*0.85;
      w2=Math.cos(ph)*0.60;
      w3=Math.tanh((x*x+z*z)-CUR_RAD)*0.45;
      w4=GAMMA*Math.sin(ph*2)*0.30;
      const isF=t.sub==="A";
      col[i*3]=isF?1.0:0.3; col[i*3+1]=isF?0.72:0.78; col[i*3+2]=isF?0.31:1.0;
    } else {
      const theta=Math.random()*Math.PI*2;
      const phi=(Math.random()-0.5)*0.5;
      const freq=Math.random()<0.3;
      const r=CUR_RAD+(freq?Math.random()*0.12:1.5+Math.random()*3.0);
      x=Math.cos(theta)*r; y=phi*2.2; z=Math.sin(theta)*r;
      w1=(Math.random()-0.5)*1.4; w2=(Math.random()-0.5)*1.0;
      w3=(Math.random()-0.5)*0.7; w4=(Math.random()-0.5)*0.4;
      col[i*3]=freq?1:0.3; col[i*3+1]=freq?0.72:0.69; col[i*3+2]=freq?0.31:1;
    }
    tokenPos4D.push([x,y,z,w1,w2,w3,w4]);
    const [px,py,pz]=project10D(x,y,z,w1,w2,w3,w4);
    pos[i*3]=px; pos[i*3+1]=py; pos[i*3+2]=pz;
  }
  const geo=new THREE.BufferGeometry();
  geo.setAttribute("position",new THREE.BufferAttribute(pos,3));
  geo.setAttribute("color",   new THREE.BufferAttribute(col,3));
  const pts=new THREE.Points(geo,new THREE.PointsMaterial({
    size:0.058,vertexColors:true,transparent:true,opacity:0.9}));
  scene.add(pts);
  tokenMeshes.push({pts,geo});
}
buildTokens();

// ── SEMANTIC EDGES ─────────────────────────────────────────────────────────
if(TOKEN_DATA.length>0){
  const fq=TOKEN_DATA.filter(t=>t.sub==="A").slice(0,90);
  fq.forEach((a,i)=>{ fq.slice(i+1,90).forEach(b=>{
    const dx=a.x-b.x,dy=a.y-b.y,dz=a.z-b.z;
    const d=Math.sqrt(dx*dx+dy*dy+dz*dz);
    if(d<0.7){
      const wa=tokenPos4D[TOKEN_DATA.indexOf(a)]||[0,0,0,0,0,0,0];
      const wb=tokenPos4D[TOKEN_DATA.indexOf(b)]||[0,0,0,0,0,0,0];
      const [pax,pay,paz]=project10D(a.x,a.y,a.z,...wa.slice(3,7));
      const [pbx,pby,pbz]=project10D(b.x,b.y,b.z,...wb.slice(3,7));
      scene.add(new THREE.Line(
        new THREE.BufferGeometry().setFromPoints([
          new THREE.Vector3(pax,pay,paz),new THREE.Vector3(pbx,pby,pbz)]),
        new THREE.LineBasicMaterial({color:0x39ffa0,transparent:true,
          opacity:0.09*(1-d/0.7)})));
    }
  });});
}

// ── BINDING ARCS — 10D Bezier through W-space ─────────────────────────────
const arcMeshes=[];
if(IS_V3456 && PHASE_DATA.length>0 && TOKEN_DATA.length>0){
  const THRESH=Math.PI/4.5;
  let cnt=0;
  const NB=Math.min(TOKEN_DATA.length,80);
  for(let a=0;a<NB&&cnt<130;a++){
    for(let b=a+1;b<NB&&cnt<130;b++){
      const pa=PHASE_DATA[a%PHASE_DATA.length];
      const pb=PHASE_DATA[b%PHASE_DATA.length];
      const dPhM=Math.min(Math.abs(pa-pb)%(Math.PI*2), Math.PI*2-Math.abs(pa-pb)%(Math.PI*2));
      if(dPhM<THRESH){
        const ta=TOKEN_DATA[a], tb=TOKEN_DATA[b];
        const wa=tokenPos4D[a]||[ta.x,ta.y,ta.z,0,0,0,0];
        const wb=tokenPos4D[b]||[tb.x,tb.y,tb.z,0,0,0,0];
        const dx=ta.x-tb.x,dy=ta.y-tb.y,dz=ta.z-tb.z;
        if(Math.sqrt(dx*dx+dy*dy+dz*dz)<1.8){
          // Arc goes through W-space — bulges in W1+W2 for depth
          const arcPts4D=[];
          for(let s=0;s<=18;s++){
            const f=s/18;
            const bulge=Math.sin(f*Math.PI);
            arcPts4D.push([
              ta.x+(tb.x-ta.x)*f,
              ta.y+(tb.y-ta.y)*f + bulge*0.35,
              ta.z+(tb.z-ta.z)*f,
              wa[3]+(wb[3]-wa[3])*f + bulge*0.55,
              wa[4]+(wb[4]-wa[4])*f + bulge*0.35,
              wa[5]+(wb[5]-wa[5])*f,
              wa[6]+(wb[6]-wa[6])*f,
            ]);
          }
          const pts2=arcPts4D.map(p=>{
            const [px,py,pz]=project10D(p[0],p[1],p[2],p[3],p[4],p[5],p[6]);
            return new THREE.Vector3(px,py,pz);
          });
          const hue=(pa/(Math.PI*2))%1;
          const col=new THREE.Color().setHSL(hue,0.9,0.62);
          const arcGeo=new THREE.BufferGeometry().setFromPoints(pts2);
          const arcMat=new THREE.LineBasicMaterial({color:col,transparent:true,
            opacity:0.30*(1-dPhM/THRESH)});
          scene.add(new THREE.Line(arcGeo,arcMat));
          arcMeshes.push({geo:arcGeo,pts4D:arcPts4D,alpha:0.30*(1-dPhM/THRESH)});
          cnt++;
        }
      }
    }
  }
}

// ── L_ALIGN SPIRAL — student↔teacher manifold bridge ─────────────────────
if(IS_V7){
  const laPts=[];
  for(let i=0;i<=120;i++){
    const f=i/120;
    const angle=f*Math.PI*4.5;
    const r=CUR_RAD*(1.0 + f*0.4);  // grows from student→teacher manifold
    const y=-2.2+f*4.4;
    const w1=Math.sin(angle*0.5)*0.5;
    const [px,py,pz]=project10D(Math.cos(angle)*r,y,Math.sin(angle)*r,w1,0,0,0);
    laPts.push(new THREE.Vector3(px,py,pz));
  }
  scene.add(new THREE.Line(
    new THREE.BufferGeometry().setFromPoints(laPts),
    new THREE.LineBasicMaterial({color:0x39ffa0,transparent:true,opacity:0.45})));
}

// ── DGW RIBBON — dimensional divergence ──────────────────────────────────
if(GW_DATA.length>1){
  const gpts=GW_DATA.map((d,i)=>{
    const f=i/GW_DATA.length;
    const angle=f*Math.PI*3;
    const r=CUR_RAD*(1.1+d.dgw*3.5);
    const w1=Math.sin(angle*0.6)*0.7;
    const w2=Math.cos(angle*0.4)*0.4;
    const [px,py,pz]=project10D(Math.cos(angle)*r,-1.8+f*0.9,Math.sin(angle)*r,w1,w2,0,0);
    return new THREE.Vector3(px,py,pz);
  });
  scene.add(new THREE.Line(
    new THREE.BufferGeometry().setFromPoints(gpts),
    new THREE.LineBasicMaterial({color:0xff9f43,transparent:true,opacity:0.55})));
}

// ── PPL CONVERGENCE HELIX ─────────────────────────────────────────────────
if(TRAIL_DATA.length>2){
  const rpts=TRAIL_DATA.map((d,i)=>{
    const f=i/TRAIL_DATA.length;
    const a=f*Math.PI*4;
    const r=Math.max(0.18,Math.min(CUR_RAD*1.9,d.ppl/600));
    const w1=Math.sin(a*0.7)*0.4;
    const [px,py,pz]=project10D(Math.cos(a)*r*2,-2.0+f*0.7,Math.sin(a)*r*2,w1,0,0,0);
    return new THREE.Vector3(px,py,pz);
  });
  scene.add(new THREE.Line(
    new THREE.BufferGeometry().setFromPoints(rpts),
    new THREE.LineBasicMaterial({color:0x080f25,transparent:true,opacity:0.5})));
}

// ── KURAMOTO LINES — 10D phase vectors ───────────────────────────────────
const kuLines=[];
for(let h=0;h<10;h++){
  const baseA=(h/10)*Math.PI*2;
  const ph=PHASE_DATA.length>h?PHASE_DATA[h]:Math.random()*Math.PI*2;
  const pts=[new THREE.Vector3(0,0,0),
    new THREE.Vector3(Math.cos(baseA)*CUR_RAD,0,Math.sin(baseA)*CUR_RAD)];
  const col=new THREE.Color().setHSL(h/10*0.7+0.5,0.9,0.60);
  const line=new THREE.Line(
    new THREE.BufferGeometry().setFromPoints(pts),
    new THREE.LineBasicMaterial({color:col,transparent:true,opacity:0.55}));
  scene.add(line);
  kuLines.push({line,baseA,phase:ph,speed:0.009+Math.random()*0.007});
}

// ── Γ RING ────────────────────────────────────────────────────────────────
if(IS_V3456 && GAMMA>0.04){
  const gr=new THREE.Mesh(
    new THREE.TorusGeometry(CUR_RAD*(0.3+GAMMA*0.7),0.018+GAMMA*0.01,8,96),
    new THREE.MeshBasicMaterial({color:0xe040fb,transparent:true,opacity:0.38+GAMMA*0.36}));
  gr.rotation.x=Math.PI/2; gr.position.y=0.3; scene.add(gr);
  const gr2=new THREE.Mesh(
    new THREE.TorusGeometry(CUR_RAD*(0.3+GAMMA*0.7),0.09,6,64),
    new THREE.MeshBasicMaterial({color:0xe040fb,transparent:true,opacity:0.05+GAMMA*0.04}));
  gr2.rotation.x=Math.PI/2; gr2.position.y=0.3; scene.add(gr2);
}

// ── TEACHER BEAM — distillation ───────────────────────────────────────────
const beamGrp=new THREE.Group();
[[0.01,5.8,0.13],[0.09,5.8,0.055],[0.28,5.8,0.025]].forEach(([ra,h,op])=>{
  const m=new THREE.Mesh(
    new THREE.CylinderGeometry(ra,0.55,h,8,1,true),
    new THREE.MeshBasicMaterial({color:0x4070ff,transparent:true,opacity:op,side:THREE.DoubleSide}));
  m.position.y=2.9; beamGrp.add(m);
});
scene.add(beamGrp);

// Distillation rain — token flow
const ND=80, dp=new Float32Array(ND*3), dProg=[];
for(let i=0;i<ND;i++){
  dp[i*3]=(Math.random()-0.5)*0.85;
  dp[i*3+1]=Math.random()*6-3;
  dp[i*3+2]=(Math.random()-0.5)*0.85;
  dProg.push(Math.random());
}
const dGeo=new THREE.BufferGeometry();
dGeo.setAttribute("position",new THREE.BufferAttribute(dp,3));
scene.add(new THREE.Points(dGeo,
  new THREE.PointsMaterial({color:0x4070ff,size:0.028,transparent:true,opacity:0.50})));

// ── DIMENSION PORTAL — abstract 10D indicator ────────────────────────────
// A nested icosahedron wireframe at center — represents 10D embedding
const icoG=new THREE.IcosahedronGeometry(0.18,0);
[[0xffffff,0.08],[0x00e5ff,0.04],[0xffd600,0.03]].forEach(([c,op],k)=>{
  const mesh=new THREE.Mesh(icoG,
    new THREE.MeshBasicMaterial({color:c,transparent:true,opacity:op,wireframe:true}));
  mesh.scale.set(1+k*0.6,1+k*0.6,1+k*0.6);
  scene.add(mesh);
});

// ── LIGHTS ─────────────────────────────────────────────────────────────────
scene.add(new THREE.AmbientLight(0x040810,3));
[{c:0x4070ff,i:2.5,d:20,p:[0,6,0]},
 {c:0xffb84f,i:1.1,d:12,p:[6,-1,4]},
 {c:0xe040fb,i:0.9,d:10,p:[-5,2,-4]},
 {c:0x00e5ff,i:0.7,d:9, p:[0,-3,5]},
 {c:0x39ffa0,i:0.5,d:8, p:[4,4,-2]},
 {c:0xff9f43,i:0.4,d:7, p:[-3,-2,3]},
].forEach(l=>{const pl=new THREE.PointLight(l.c,l.i,l.d);pl.position.set(...l.p);scene.add(pl);});

// ── ORBIT CONTROLS ─────────────────────────────────────────────────────────
let isDrag=false, prevM={x:0,y:0};
let sph={theta:0,phi:Math.PI/3.8,radius:10};
let autoOrbit=true;

function updateCam(){
  const r=sph.radius;
  const ph=Math.max(0.05,Math.min(Math.PI-0.05,sph.phi));
  camera.position.x=r*Math.sin(ph)*Math.sin(sph.theta);
  camera.position.y=r*Math.cos(ph)+0.5;
  camera.position.z=r*Math.sin(ph)*Math.cos(sph.theta);
  camera.lookAt(0,0.5,0);
}
canvas.addEventListener('mousedown',e=>{isDrag=true;autoOrbit=false;prevM={x:e.clientX,y:e.clientY};});
canvas.addEventListener('mousemove',e=>{
  if(!isDrag)return;
  sph.theta-=(e.clientX-prevM.x)*0.007;
  sph.phi  -=(e.clientY-prevM.y)*0.007;
  sph.phi=Math.max(0.08,Math.min(Math.PI-0.08,sph.phi));
  prevM={x:e.clientX,y:e.clientY}; updateCam();
});
canvas.addEventListener('mouseup',()=>isDrag=false);
canvas.addEventListener('wheel',e=>{
  sph.radius=Math.max(3,Math.min(22,sph.radius+e.deltaY*0.014));
  updateCam(); e.preventDefault();},{passive:false});
canvas.addEventListener('touchstart',e=>{prevM={x:e.touches[0].clientX,y:e.touches[0].clientY};autoOrbit=false;},{passive:false});
canvas.addEventListener('touchmove',e=>{
  sph.theta-=(e.touches[0].clientX-prevM.x)*0.006;
  sph.phi  -=(e.touches[0].clientY-prevM.y)*0.006;
  sph.phi=Math.max(0.08,Math.min(Math.PI-0.08,sph.phi));
  prevM={x:e.touches[0].clientX,y:e.touches[0].clientY}; updateCam(); e.preventDefault();
},{passive:false});
window.addEventListener('keydown',e=>{
  if(e.code==='Space'){wPaused=!wPaused;e.preventDefault();}
  if(e.code==='KeyR'){sph={theta:0,phi:Math.PI/3.8,radius:10};autoOrbit=true;updateCam();}
  if(e.key==='1') wSpeeds=[0.0055,0.0038,0.0021,0.0013];
  if(e.key==='2') wSpeeds=[0.011,0.0076,0.0042,0.0026];
  if(e.key==='3') wSpeeds=[0.022,0.015,0.008,0.005];
  if(e.key==='4') wSpeeds=[0.0014,0.001,0.0006,0.0003];
});

// ── ANIMATE ────────────────────────────────────────────────────────────────
let t=0, last=performance.now(), fc=0, ft=0;
const wEl=document.getElementById("w-angles");
const fEl=document.getElementById("fps-ctr");

function animate(now){
  requestAnimationFrame(animate);
  const dt=(now-last)/1000; last=now; t+=dt; fc++; ft+=dt;
  if(ft>=1){fEl.textContent=Math.round(fc/ft)+" fps";fc=0;ft=0;}

  // W-axes evolution
  if(!wPaused) wAngles=wAngles.map((w,i)=>w+wSpeeds[i]*dt*60);
  if(wEl) wEl.textContent=`θ₁=${wAngles[0].toFixed(2)} θ₂=${wAngles[1].toFixed(2)}`;

  // Update token positions via 10D rotation
  tokenMeshes.forEach(tm=>{
    const arr=tm.geo.attributes.position.array;
    tokenPos4D.forEach(([x,y,z,w1,w2,w3,w4],i)=>{
      const [rx,ry,rz,rw1]=rotateW1(x,y,z,w1,wAngles[0]);
      const [rx2,ry2,rz2,rw2]=rotateW2(rx,ry,rz,w2,wAngles[1]);
      const [rx3,ry3,rz3,rw3]=rotateW3(rx2,ry2,rz2,w3,wAngles[2]);
      const [px,py,pz]=project10D(rx3,ry3,rz3,rw1,rw2,rw3,w4);
      arr[i*3]=px; arr[i*3+1]=py; arr[i*3+2]=pz;
    });
    tm.geo.attributes.position.needsUpdate=true;
  });

  // Update binding arcs 10D
  arcMeshes.forEach(arc=>{
    const pts=arc.pts4D.map(p=>{
      const [rx,ry,rz,rw1]=rotateW1(p[0],p[1],p[2],p[3],wAngles[0]);
      const [rx2,ry2,rz2,rw2]=rotateW2(rx,ry,rz,p[4],wAngles[1]);
      const [px,py,pz]=project10D(rx2,ry2,rz2,rw1,rw2,p[5],p[6]);
      return new THREE.Vector3(px,py,pz);
    });
    arc.geo.setFromPoints(pts);
    arc.geo.attributes.position.needsUpdate=true;
  });

  // Auto orbit
  if(autoOrbit){
    sph.theta+=0.0016; sph.phi=Math.PI/3.8+Math.sin(t*0.045)*0.14; updateCam();
  }

  // Shell pulse — K_eff modulates amplitude
  const kMod=IS_V7?0.5+KEFF*1.5:1.0;
  const pulse=1+Math.sin(t*1.7)*0.011*kMod;
  shellS.scale.set(pulse,pulse,pulse); shellSg.scale.set(pulse*1.05,pulse*1.02,pulse*1.05);
  shellT.scale.set(1.02+Math.sin(t*0.9)*0.008,1.01,1.02+Math.sin(t*0.9)*0.008);

  // Pacemaker rings — each pulses at its characteristic frequency
  pacRings.forEach((pr,k)=>{
    const freq=[1.3,1.7,2.1][k];
    const amp=0.015+GAMMA*0.02;
    const sc=1+Math.sin(t*freq+pr.phase)*amp;
    pr.rm.scale.set(sc,1,sc);
    pr.rg.scale.set(sc*1.02,1,sc*1.02);
    pr.rm.material.opacity=0.40+Math.sin(t*freq+pr.phase)*0.20;
  });

  // Kuramoto evolution — 10 lines
  const frust=IS_V3456?0.26+GAMMA*0.16:0.20;
  kuLines.forEach(kl=>{
    kl.phase+=kl.speed+CUR_RDC*0.003;
    const angle=kl.baseA+Math.sin(kl.phase)*frust;
    const pa=kl.line.geometry.attributes.position;
    const py=Math.sin(kl.phase)*0.25;
    const pr2=CUR_RAD*(0.85+KEFF*0.15);
    pa.setXYZ(1,Math.cos(angle)*pr2,py,Math.sin(angle)*pr2);
    pa.needsUpdate=true;
  });

  // Distillation rain
  const dArr=dGeo.attributes.position.array;
  for(let i=0;i<ND;i++){
    dProg[i]=(dProg[i]+0.0022)%1;
    dArr[i*3+1]=6.5-dProg[i]*10;
  }
  dGeo.attributes.position.needsUpdate=true;
  beamGrp.children.forEach((b,k)=>{
    b.material.opacity=[0.13,0.055,0.025][k]*(0.65+Math.sin(t*1.4)*0.35);
  });

  // Icosahedron portal rotation
  scene.children.forEach(c=>{
    if(c.geometry && c.geometry.type==='IcosahedronGeometry')
      c.rotation.y+=0.006; c.rotation.x+=0.003;
  });

  renderer.render(scene,camera);
}
updateCam(); animate(performance.now());
})();
</script></body></html>
"""

In [ ]:
"""Cell 3 — HyDRA Dimension 10 Visualizer"""
import json as _json
from IPython.display import HTML

def build_d10_viz(height=860):
    train   = read_csv(TRAIN_CSV)
    val     = read_csv(VAL_CSV)
    binding = read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7") else []
    gw_rows = read_csv(GW_CSV)      if RUN_MODE in ("v6","v7") else []
    snap    = load_snapshot()
    m       = latest_metrics(train, val) if train else {}
    bm      = latest_binding(binding)

    tokens_json = _json.dumps(snap.get("tokens", []))
    phases_json = _json.dumps(snap.get("phases", bm.get("phases", [])))
    trail       = train[-600:] if len(train)>600 else train
    trail_json  = _json.dumps([{"step":int(r.get("step",0) or 0),
                                 "ppl":min(float(r.get("ppl",999) or 999),3000)}
                                for r in trail])
    gw_json = _json.dumps([{"step":int(r["step"]),"dgw":float(r.get("dgw",0) or 0)}
                            for r in gw_rows])

    gamma    = bm.get("order_param", 0.0)
    cur_rad  = m.get("rad", 1.5)
    cur_rdc  = m.get("rdc", 0.1)
    cur_lstd = m.get("logit_std", 0.0)
    keff     = m.get("k_eff", 0.3)
    lalign   = m.get("l_align", 0.0)
    is_v7    = "true" if RUN_MODE == "v7" else "false"
    is_v3456 = "true" if RUN_MODE in ("v3","v4","v5","v6","v7") else "false"

    html = VIZ_HTML.replace("{{HEIGHT}}", str(height))
    html = html.replace("{{LABEL}}", RUN_LABEL[:22])
    html = html.replace("{{STEP}}", f"{m.get('step',0):,}")
    html = html.replace("{{PPL}}", f"{m.get('ppl',9999):.1f}")
    html = html.replace("{{RDC}}", f"{m.get('rdc',0):.3f}")
    html = html.replace("{{RAD}}", f"{cur_rad:.3f}")
    html = html.replace("{{LSTD}}", f"{cur_lstd:.3f}")
    html = html.replace("{{DIV}}", f"{m.get('div',0):.3f}")
    html = html.replace("{{WENT}}", f"{m.get('w_ent',0):.3f}")
    html = html.replace("{{GAMMA}}", f"{gamma:.3f}")
    html = html.replace("{{KEFF}}", f"{keff:.3f}")
    html = html.replace("{{LALIGN}}", f"{lalign:.4f}")
    html = html.replace("{{VALPPL}}", f"{m.get('val_ppl',9999):.1f}")
    html = html.replace("{{VALSTEP}}", f"{m.get('val_step',0):,}")
    html = html.replace("{{TOKEN_DATA}}", tokens_json)
    html = html.replace("{{PHASE_DATA}}", phases_json)
    html = html.replace("{{TRAIL_DATA}}", trail_json)
    html = html.replace("{{GW_DATA}}", gw_json)
    html = html.replace("{{GAMMA_VAL}}", f"{gamma:.4f}")
    html = html.replace("{{CUR_RAD}}", f"{cur_rad:.4f}")
    html = html.replace("{{CUR_RDC}}", f"{cur_rdc:.4f}")
    html = html.replace("{{CUR_LSTD}}", f"{cur_lstd:.3f}")
    html = html.replace("{{KEFF_VAL}}", f"{keff:.4f}")
    html = html.replace("{{IS_V7}}", is_v7)
    html = html.replace("{{IS_V3456}}", is_v3456)

    gamma_badge = f'<div class="gamma">Γ={gamma:.2f}</div>' if gamma>0.05 else ""
    keff_badge  = f'<div class="keff">K_eff={keff:.3f}</div>' if RUN_MODE in ("v7",) else ""
    lalign_badge= f'<div class="lalign">L_align={lalign:.4f}</div>' if lalign>0 else ""
    html = html.replace("{{GAMMA_BADGE}}", gamma_badge)
    html = html.replace("{{KEFF_BADGE}}", keff_badge)
    html = html.replace("{{LALIGN_BADGE}}", lalign_badge)

    return HTML(html)

display(build_d10_viz())


In [ ]:
"""Cell 4 — Full Metrics Panel (v7-aware)"""
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def render_full_metrics_v7():
    train   = read_csv(TRAIN_CSV); val = read_csv(VAL_CSV)
    binding = read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7") else []
    gw_rows = read_csv(GW_CSV)      if RUN_MODE in ("v6","v7") else []
    if not train: print("⏳ Aguardando dados..."); return
    m  = latest_metrics(train, val)
    bm = latest_binding(binding)

    s   = [int(r["step"])           for r in train]
    pp  = [float(r["ppl"])          for r in train if r.get("ppl") and float(r["ppl"])<5000]
    lo  = [float(r["loss"])         for r in train if r.get("loss")]
    rc  = [float(r["rdc_ema"])      for r in train if r.get("rdc_ema")]
    rd  = [float(r["mean_radius"])  for r in train if r.get("mean_radius")]
    ls  = [float(r["logit_std"])    for r in train if r.get("logit_std")]
    we  = [float(r["w_entropy"])    for r in train if r.get("w_entropy")]
    hi  = [float(r["l_hidden"])     for r in train if r.get("l_hidden")]
    dv  = [float(r["diversity"])    for r in train if r.get("diversity")]
    ke  = [float(r["k_eff"])        for r in train if r.get("k_eff")]
    la  = [float(r["l_align"])      for r in train if r.get("l_align")]
    vs  = [int(r["step"])           for r in val]
    vp  = [float(r["val_ppl"])      for r in val  if r.get("val_ppl")]
    bs  = [int(r["step"])                    for r in binding]
    bg  = [float(r.get("order_param",0) or 0) for r in binding]
    gws = [int(r["step"])           for r in gw_rows]
    gwd = [float(r.get("dgw",0))   for r in gw_rows]

    is_v7 = RUN_MODE in ("v7",)
    n_rows = 5 if is_v7 else (4 if RUN_MODE in ("v3","v4","v5","v6") else 3)

    fig = make_subplots(rows=n_rows, cols=2,
        subplot_titles=["PPL","Loss","RDC","Radius","logit_std","w_entropy",
                        "Γ Order Param","hidden align",
                        "K_eff (Kuramoto Gate)","L_align"][:n_rows*2],
        vertical_spacing=0.08, horizontal_spacing=0.07)

    kw=dict(mode="lines",showlegend=False)
    fig.add_trace(go.Scatter(x=s[:len(pp)],y=pp,line=dict(color="#4f8fff",width=1.5),**kw),row=1,col=1)
    if vp: fig.add_trace(go.Scatter(x=vs[:len(vp)],y=vp,mode="lines+markers",
        line=dict(color="#39ffa0",width=2),marker=dict(size=4),showlegend=False),row=1,col=1)
    fig.add_trace(go.Scatter(x=s[:len(lo)],y=lo, line=dict(color="#b07fff",width=1.5),**kw),row=1,col=2)
    fig.add_trace(go.Scatter(x=s[:len(rc)],y=rc, line=dict(color="#ff4060",width=1.5),**kw),row=2,col=1)
    fig.add_hline(y=2.0,line_dash="dash",line_color="#ff4060",row=2,col=1)
    fig.add_trace(go.Scatter(x=s[:len(rd)],y=rd, line=dict(color="#ffb84f",width=1.5),**kw),row=2,col=2)
    fig.add_hline(y=1.5,line_dash="dash",line_color="#ffb84f",row=2,col=2)
    fig.add_trace(go.Scatter(x=s[:len(ls)],y=ls, line=dict(color="#00e5ff",width=1.5),**kw),row=3,col=1)
    fig.add_hline(y=3.0,line_dash="dash",line_color="#ff4060",annotation_text="ceiling",row=3,col=1)
    fig.add_trace(go.Scatter(x=s[:len(we)],y=we, line=dict(color="#39ffa0",width=1.5),**kw),row=3,col=2)
    if n_rows>=4:
        if bs: fig.add_trace(go.Scatter(x=bs,y=bg,line=dict(color="#e040fb",width=2),**kw),row=4,col=1)
        fig.add_hline(y=0.3,line_dash="dash",line_color="#e040fb",row=4,col=1)
        fig.add_trace(go.Scatter(x=s[:len(hi)],y=hi,line=dict(color="#aaffcc",width=1.5),**kw),row=4,col=2)
    if is_v7 and n_rows==5:
        if ke: fig.add_trace(go.Scatter(x=s[:len(ke)],y=ke,line=dict(color="#00e5ff",width=2),**kw),row=5,col=1)
        fig.add_hline(y=0.05,line_dash="dot",line_color="#00e5ff",annotation_text="K_min",row=5,col=1)
        fig.add_hline(y=0.30,line_dash="dot",line_color="#00e5ff",annotation_text="K_max",row=5,col=1)
        if la: fig.add_trace(go.Scatter(x=s[:len(la)],y=la,line=dict(color="#39ffa0",width=2),**kw),row=5,col=2)
        # Pacemaker activation markers
        for pac_step, pac_col, pac_name in [(0,"#00e5ff","L4"),(0,"#ffd600","L8"),(0,"#e040fb","L11")]:
            fig.add_vline(x=pac_step,line_dash="dot",line_color=pac_col,
                annotation_text=pac_name,row=5,col=1)

    fig.update_layout(
        height=950 if n_rows==5 else (780 if n_rows==4 else 600),
        paper_bgcolor="#000105", plot_bgcolor="#05080f",
        font=dict(color="#c8d4f0",size=10), showlegend=False,
        title=dict(
            text=(f"{RUN_LABEL} · Step {m['step']:,}/200k · PPL {m['ppl']:.1f} · "
                  f"rdc {m['rdc']:.3f} · Γ={bm['order_param']:.3f} · "
                  f"K_eff={m['k_eff']:.3f} · L_align={m['l_align']:.4f}"),
            font=dict(size=11,color="#fff")),
        margin=dict(t=55,b=15,l=40,r=20))
    for i in range(1,n_rows+1):
        for j in range(1,3):
            fig.update_xaxes(gridcolor="#0a1220",row=i,col=j)
            fig.update_yaxes(gridcolor="#0a1220",row=i,col=j)
    fig.show()

render_full_metrics_v7()


⏳ Aguardando dados...


In [ ]:
"""Cell 5 — Live Loop + Background MP4"""
import time, threading, base64
import numpy as _np
from IPython.display import clear_output, display, HTML
import os; os.system("pip install -q imageio[ffmpeg] imageio-ffmpeg pillow kaleido 2>/dev/null")
import imageio
from PIL import Image as PILImage
import plotly.io as pio

REFRESH       = 90
CAPTURE_EVERY = 120
VIDEO_FPS     = 4
VIDEO_OUT     = TIMELAPSE_DIR / "timelapse_d10.mp4"
MP4_LIVE      = TIMELAPSE_DIR / "live_d10.mp4"
GIF_PREVIEW   = TIMELAPSE_DIR / "preview_d10.gif"

_stop          = threading.Event()
_frames        = [0]
_last_cap      = [0.0]
_mp4_cnt       = [0]
_mp4_writer    = [None]
_mp4_lock      = threading.Lock()

def _capture_frame():
    train=read_csv(TRAIN_CSV); val=read_csv(VAL_CSV)
    if not train: return None
    m  =latest_metrics(train,val)
    bm =latest_binding(read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7") else [])
    gw =read_csv(GW_CSV) if RUN_MODE in ("v6","v7") else []
    s  =[int(r["step"])          for r in train]
    pp =[float(r["ppl"])         for r in train if r.get("ppl") and float(r["ppl"])<5000]
    rc =[float(r["rdc_ema"])     for r in train if r.get("rdc_ema")]
    ls =[float(r["logit_std"])   for r in train if r.get("logit_std")]
    dv =[float(r["diversity"])   for r in train if r.get("diversity")]
    ke =[float(r["k_eff"])       for r in train if r.get("k_eff")]
    la =[float(r["l_align"])     for r in train if r.get("l_align")]
    vs =[int(r["step"])          for r in val]
    vp =[float(r["val_ppl"])     for r in val  if r.get("val_ppl")]
    bs =[int(r["step"])                    for r in read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7")]
    bg =[float(r.get("order_param",0) or 0) for r in read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7")]
    gws=[int(r["step"])          for r in gw]
    gwd=[float(r.get("dgw",0))  for r in gw]

    n_rows=5 if RUN_MODE=="v7" else (4 if RUN_MODE in ("v3","v4","v5","v6") else 2)
    from plotly.subplots import make_subplots
    fig=make_subplots(rows=n_rows,cols=2,vertical_spacing=0.1,horizontal_spacing=0.08,
        subplot_titles=["PPL","Loss","RDC","logit_std","Γ","hidden",
                        "K_eff","L_align"][:n_rows*2])
    kw=dict(mode="lines",showlegend=False)
    lo=[float(r["loss"]) for r in train if r.get("loss")]
    fig.add_trace(go.Scatter(x=s[:len(pp)],y=pp,line=dict(color="#4f8fff",width=1.5),**kw),row=1,col=1)
    if vp: fig.add_trace(go.Scatter(x=vs[:len(vp)],y=vp,mode="lines+markers",
        line=dict(color="#39ffa0",width=2),marker=dict(size=3),showlegend=False),row=1,col=1)
    fig.add_trace(go.Scatter(x=s[:len(lo)],y=lo,line=dict(color="#b07fff",width=1.5),**kw),row=1,col=2)
    fig.add_trace(go.Scatter(x=s[:len(rc)],y=rc,line=dict(color="#ff4060",width=1.5),**kw),row=2,col=1)
    fig.add_hline(y=2.0,line_dash="dash",line_color="#ff4060",row=2,col=1)
    fig.add_trace(go.Scatter(x=s[:len(ls)],y=ls,line=dict(color="#00e5ff",width=1.5),**kw),row=2,col=2)
    fig.add_hline(y=3.0,line_dash="dash",line_color="#ff4060",row=2,col=2)
    if n_rows>=3 and bs:
        fig.add_trace(go.Scatter(x=bs,y=bg,line=dict(color="#e040fb",width=2),**kw),row=3,col=1)
        hi=[float(r.get("l_hidden",0)) for r in train if r.get("l_hidden")]
        fig.add_trace(go.Scatter(x=s[:len(hi)],y=hi,line=dict(color="#aaffcc",width=1.5),**kw),row=3,col=2)
    if n_rows==5:
        if ke: fig.add_trace(go.Scatter(x=s[:len(ke)],y=ke,line=dict(color="#00e5ff",width=2),**kw),row=4,col=1)
        if la: fig.add_trace(go.Scatter(x=s[:len(la)],y=la,line=dict(color="#39ffa0",width=2),**kw),row=4,col=2)
        dv2=[float(r["diversity"]) for r in train if r.get("diversity")]
        fig.add_trace(go.Scatter(x=s[:len(dv2)],y=dv2,line=dict(color="#ffd600",width=2),**kw),row=5,col=1)
        if gwd: fig.add_trace(go.Scatter(x=gws,y=gwd,line=dict(color="#ff9f43",width=2),**kw),row=5,col=2)

    mp4_sz=MP4_LIVE.stat().st_size/1e6 if MP4_LIVE.exists() else 0
    fig.update_layout(
        width=1280,height=780 if n_rows==5 else 560,
        paper_bgcolor="#000105",plot_bgcolor="#05080f",
        font=dict(color="#c8d4f0",size=9),showlegend=False,
        title=dict(text=(f"{RUN_LABEL} · Step {m['step']:,} · PPL {m['ppl']:.1f} · "
                         f"rdc {m['rdc']:.3f} · Γ={bm['order_param']:.3f} · "
                         f"🎬{_mp4_cnt[0]}fr/{mp4_sz:.1f}MB · {time.strftime('%H:%M:%S')}"),
                  font=dict(size=10,color="#fff")),
        margin=dict(t=45,b=12,l=35,r=20))
    for i in range(1,n_rows+1):
        for j in range(1,3):
            fig.update_xaxes(gridcolor="#0a1220",row=i,col=j)
            fig.update_yaxes(gridcolor="#0a1220",row=i,col=j)
    try:
        png=pio.to_image(fig,format="png",width=1280,height=780 if n_rows==5 else 560)
        fp=TIMELAPSE_DIR/f"frame_{m['step']:08d}.png"
        fp.write_bytes(png); return fp
    except: return None

def _open_mp4():
    try:
        w=imageio.get_writer(str(MP4_LIVE),fps=VIDEO_FPS,codec="libx264",
                             quality=8,pixelformat="yuv420p",macro_block_size=None)
        _mp4_writer[0]=w; print(f"  🎬 MP4 → {MP4_LIVE.name}")
    except Exception as e: print(f"  ⚠️ MP4: {e}")

def _append_frame(fp):
    if _mp4_writer[0] is None: return
    try:
        img=_np.array(PILImage.open(fp).convert("RGB"))
        with _mp4_lock: _mp4_writer[0].append_data(img)
        _mp4_cnt[0]+=1
    except: pass

def _close_mp4():
    with _mp4_lock:
        if _mp4_writer[0] is not None:
            try: _mp4_writer[0].close()
            except: pass
            _mp4_writer[0]=None

def _compile_gif(fps=3,max_frames=20):
    frs=sorted(TIMELAPSE_DIR.glob("frame_*.png"))[-max_frames:]
    if len(frs)<2: return
    imgs=[_np.array(PILImage.open(f).convert("RGB").resize((900,560))) for f in frs]
    imageio.mimsave(str(GIF_PREVIEW),imgs,duration=1.0/fps,loop=0)

_last_gif=[0]; GIF_EVERY=5

def _tl():
    time.sleep(3); _open_mp4()
    while not _stop.is_set():
        if time.time()-_last_cap[0]>=CAPTURE_EVERY:
            try:
                fp=_capture_frame()
                if fp:
                    _frames[0]+=1; _append_frame(fp)
                    if _mp4_cnt[0]%5==0:
                        sz=MP4_LIVE.stat().st_size/1e6 if MP4_LIVE.exists() else 0
                        print(f"  [🎬] {_mp4_cnt[0]}fr · {sz:.1f}MB")
                _last_cap[0]=time.time()
            except Exception as e: print(f"  ⚠️ {e}")
        time.sleep(5)

def _gif_w():
    time.sleep(30)
    while not _stop.is_set():
        if _frames[0]>=2 and _frames[0]-_last_gif[0]>=GIF_EVERY:
            try: _compile_gif(); _last_gif[0]=_frames[0]
            except: pass
        time.sleep(15)

_t1=threading.Thread(target=_tl,daemon=True,name="mp4-rec"); _t1.start()
_t2=threading.Thread(target=_gif_w,daemon=True,name="gif-cmp"); _t2.start()
print(f"🚀 {RUN_LABEL} | 📸/{CAPTURE_EVERY}s · 🎬 live MP4 · 🖥️/{REFRESH}s")

import plotly.graph_objects as go
try:
    while True:
        clear_output(wait=True)
        train=read_csv(TRAIN_CSV); val=read_csv(VAL_CSV)
        m=latest_metrics(train,val) if train else {}
        bm=latest_binding(read_csv(BINDING_CSV) if RUN_MODE in ("v3","v4","v5","v6","v7") else [])
        gw_rows=read_csv(GW_CSV) if RUN_MODE in ("v6","v7") else []
        gw_last=float(gw_rows[-1]["dgw"]) if gw_rows else 0.0
        nxt=max(0,int(CAPTURE_EVERY-(time.time()-_last_cap[0])))
        mp4_sz=MP4_LIVE.stat().st_size/1e6 if MP4_LIVE.exists() else 0
        print(f"[{time.strftime('%H:%M:%S')}] step={m.get('step',0):,} "
              f"ppl={m.get('ppl',9999):.1f} rdc={m.get('rdc',0):.3f} "
              f"Γ={bm['order_param']:.3f} DGW={gw_last:.4f} "
              f"K_eff={m.get('k_eff',0):.3f} L_align={m.get('l_align',0):.4f} | "
              f"📸{_frames[0]} · 🎬{_mp4_cnt[0]}fr/{mp4_sz:.1f}MB · next:{nxt}s")
        if GIF_PREVIEW.exists():
            gb=base64.b64encode(GIF_PREVIEW.read_bytes()).decode()
            gk=GIF_PREVIEW.stat().st_size/1024
            display(HTML(f'''<div style="text-align:center;margin:4px 0;">
              <div style="font-family:'Syne Mono',monospace;font-size:8px;color:#1e2d50">
                🎞️ {_frames[0]} frames · {gk:.0f}KB</div>
              <img src="data:image/gif;base64,{gb}"
                style="width:100%;max-width:1000px;border-radius:8px;
                       border:1px solid #0d1a35;"/></div>'''))
        display(build_d10_viz(height=780))
        render_full_metrics_v7()
        time.sleep(REFRESH)
except KeyboardInterrupt:
    print(f"\n⏹️  {_frames[0]} frames · MP4: {_mp4_cnt[0]} frames")
    _stop.set(); _close_mp4(); _t1.join(8); _t2.join(8)
    if MP4_LIVE.exists():
        print(f"✅ live: {MP4_LIVE} ({MP4_LIVE.stat().st_size/1e6:.1f}MB)")
    if _frames[0]>=2:
        try:
            w=imageio.get_writer(str(VIDEO_OUT),fps=VIDEO_FPS,codec="libx264",quality=9,pixelformat="yuv420p")
            for f in sorted(TIMELAPSE_DIR.glob("frame_*.png")):
                w.append_data(_np.array(PILImage.open(f).convert("RGB")))
            w.close()
            print(f"✅ final: {VIDEO_OUT} ({VIDEO_OUT.stat().st_size/1e6:.1f}MB)")
        except Exception as e: print(f"⚠️ {e}")
        _compile_gif(fps=VIDEO_FPS,max_frames=9999)
        print(f"✅ gif: {GIF_PREVIEW}")


[02:24:08] step=1,603 ppl=450.2 rdc=0.214 Γ=0.000 DGW=0.0000 K_eff=0.300 L_align=0.0000 | 📸0 · 🎬0fr/0.0MB · next:56s


In [ ]:
"""Cell 6 — Playback"""
from IPython.display import Video, display
if VIDEO_OUT.exists():
    print(f"✅ {VIDEO_OUT.stat().st_size/1e6:.1f}MB")
    display(Video(str(VIDEO_OUT),embed=True,width=1280))
else:
    frs=sorted(TIMELAPSE_DIR.glob("frame_*.png"))
    print(f"⏳ {len(frs)} frames — Ctrl+C na Cell 5 para compilar")
